In [14]:
# --- Importaciones ---
import requests
from bs4 import BeautifulSoup
# import pandas as pd
import time
import json # Para imprimir diccionarios de forma legible

# --- Instalaciones (si es necesario) ---


# Clase 2: Conferencia Práctica - Scraping Estático (requests y BeautifulSoup)
# 
# **Objetivo:** Dominar la lectura de HTML y la extracción de datos estructurados (títulos, precios, calificaciones) de un sitio web estático.
# 
# 
# **Herramientas:**
# 1.  `wget` (Línea de comandos)
# 2.  `requests` (El "Mensajero" de Python - Con manejo de errores)
# 3.  `BeautifulSoup` (El "Navegador/Intérprete" de HTML - Métodos `find` y `select`)
# 4.  `pandas` (El "Contenedor" de Datos)



# Parte 1: El Protocolo - `wget` y `requests` (Confiabilidad y User-Agent)
# 
# Necesitamos "pedir" el documento a través del protocolo HTTP.

# A. El Clásico: `wget`
# 
# `wget` es una herramienta de terminal que simplemente descarga un recurso.
# 
# **Nota:** En un notebook, usamos `!` para ejecutar comandos de terminal.


In [15]:
# Descargamos el 'index.html' del sitio de libros
!wget -O libros.html http://books.toscrape.com/
print("¡Archivo 'libros.html' descargado! Revisa el explorador de archivos.")

--2025-11-14 09:36:41--  http://books.toscrape.com/
Resolving books.toscrape.com (books.toscrape.com)... 

35.211.122.109
Connecting to books.toscrape.com (books.toscrape.com)|35.211.122.109|:80... failed: Connection timed out.
Retrying.

--2025-11-14 09:38:58--  (try: 2)  http://books.toscrape.com/
Connecting to books.toscrape.com (books.toscrape.com)|35.211.122.109|:80... failed: Connection timed out.
Retrying.

--2025-11-14 09:41:14--  (try: 3)  http://books.toscrape.com/
Connecting to books.toscrape.com (books.toscrape.com)|35.211.122.109|:80... failed: Connection timed out.
Retrying.

--2025-11-14 09:43:30--  (try: 4)  http://books.toscrape.com/
Connecting to books.toscrape.com (books.toscrape.com)|35.211.122.109|:80... failed: Connection timed out.
Retrying.

--2025-11-14 09:45:50--  (try: 5)  http://books.toscrape.com/
Connecting to books.toscrape.com (books.toscrape.com)|35.211.122.109|:80... failed: Connection timed out.
Retrying.

--2025-11-14 09:48:10--  (try: 6)  http://books.toscrape.com/
Connecting to books.toscrape.com (books.toscrape.com)|35.211.122.109|:80... failed: Connec

# **Conclusión de `wget`:** Útil para descargar, pero el procesamiento debe ocurrir *dentro* de Python.

#  B. El Estándar de Python: `requests` y el **User-Agent**
# 
# `requests` hace la petición HTTP. Para ser robustos, debemos manejar posibles fallos de red o tiempos de espera.

# **¿Qué es el User-Agent?**
# 
# 1.  **Definición:** El `User-Agent` es una cadena de texto que se envía en el encabezado (Header) de la petición HTTP. Su función es decirle al servidor **quién es el cliente que hace la solicitud** (por ejemplo, qué navegador, qué sistema operativo o qué robot es).
# 
# 2.  **¿Por qué es importante?**
#     * El servidor lo usa para servir el contenido correcto (ej. una página optimizada para móvil o para desktop).
#     * **Permite la identificación.**
# 
# 3.  **¿Qué pasa si NO lo usamos (o usamos el default de `requests`)?**
#     * El `User-Agent` por defecto de `requests` es fácilmente identificable como un script automatizado de Python.
#     * **Consecuencias Comunes:**
#         * **Bloqueo (403 Forbidden):** El servidor puede negar la petición inmediatamente si identifica el `User-Agent` por defecto de Python como un bot no deseado.
#         * **Contenido Mínimo:** Algunos sitios sirven contenido parcial o nulo a los bots conocidos para ahorrar recursos.
#         * **Límites de Tasa (429 Too Many Requests):** Al no identificarnos, el servidor asume que somos un robot agresivo y nos impone límites de velocidad o un bloqueo temporal.
#     * **Buena Práctica:** Siempre envía un `User-Agent` que simule un navegador común (para sitios que lo requieren) o uno descriptivo que identifique tu bot (para fines éticos).

In [16]:
URL = "http://books.toscrape.com/"
HEADERS = {
    # Imitamos un navegador común de Windows para asegurar el acceso, aunque lo ideal es identificar nuestro bot
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
TIMEOUT = 5 # Definimos un tiempo máximo de espera de 5 segundos

print(f"Haciendo petición GET a {URL} con un timeout de {TIMEOUT}s...")
response = None
html_content = ""

try:
    # Hacemos la petición GET, agregando headers y timeout
    response = requests.get(URL, headers=HEADERS, timeout=TIMEOUT)

    # ¡Validación del estado HTTP! (Código 200)
    if response.status_code == 200:
        print("¡Éxito! El servidor respondió OK (200).")
        html_content = response.text
    else:
        print(f"¡Error HTTP! El servidor respondió con el código: {response.status_code}")
        
except requests.exceptions.Timeout:
    print(f"Error de tiempo de espera (Timeout) después de {TIMEOUT} segundos.")
except requests.exceptions.RequestException as e:
    print(f"Error de conexión: {e}")
finally:
    if not html_content:
        print("El contenido HTML está vacío. No se puede continuar con el parseo.")


Haciendo petición GET a http://books.toscrape.com/ con un timeout de 5s...
¡Éxito! El servidor respondió OK (200).


# Parte 2: La Estructura - Entendiendo HTML y el DOM (Conferencia Extendida)
# 
# El corazón del scraping estático es saber dónde buscar en el documento HTML que acabamos de recibir.

# El HTML (HyperText Markup Language) es el lenguaje de marcado que describe la estructura de una página web. Se basa en tres componentes clave que debes dominar:
# 
# 1. Etiquetas (Tags)
# * **Definición:** Son los nombres de los elementos encerrados en corchetes angulares (`< >`). Definen el tipo de contenido o función.
# * **Ejemplos Comunes:**
#     * `<div>`: Un contenedor genérico (la caja principal de la estructura).
#     * `<h1>` a `<h6>`: Títulos y subtítulos.
#     * `<p>`: Párrafos de texto.
#     * `<a>`: Enlaces o hipervínculos (`Anchor`).
#     * `<img>`: Imágenes.
#     * `<article>`: Un bloque de contenido independiente (usado para cada libro en Books to Scrape).
# * **Estructura:** Casi todas las etiquetas tienen una de apertura y una de cierre: `<p>Contenido</p>`.
# 
# #### 2. Atributos
# * **Definición:** Son propiedades que se añaden a la etiqueta de apertura para proporcionar información adicional sobre el elemento (no visible en la página).
# * **Sintaxis:** Se escriben como `atributo="valor"` dentro de la etiqueta.
# * **Atributos Fundamentales para el Scraping:**
#     * **`class`:** El atributo más crucial. Se utiliza para aplicar estilos (CSS) a **múltiples** elementos.
#         * **Ejemplo:** `<div class="product_pod price_color">`
#         * **Regla:** Un elemento puede tener múltiples clases (`product_pod` y `price_color`).
#     * **`id`:** Un identificador **único** que solo debería usarse una vez por página.
#         * **Ejemplo:** `<div id="navbar-top">`
#     * **`href`:** El destino del enlace, solo usado en etiquetas `<a>`.
#         * **Ejemplo:** `<a href="/catalogue/...">`
#     * **`src`:** La fuente (URL) de un recurso, usado en etiquetas `<img>` o `<script>`.
#         * **Ejemplo:** `<img src="/media/cache/2c0570a...jpg">`
# 
# #### 3. Contenido
# * **Definición:** El texto o los elementos anidados que se encuentran entre las etiquetas de apertura y cierre. Es el dato que queremos extraer.
# * **Ejemplo:** `<h1>El Título Principal</h1>` -> El contenido es *"El Título Principal"*.
# 
# ---
# 
# ### **CONFERENCIA: El DOM (Document Object Model) y la Navegación**
# 
# **El DOM (Document Object Model)** es la representación del HTML como un árbol jerárquico. `BeautifulSoup` convierte el string de HTML en este árbol de objetos de Python, lo que permite la navegación lógica en lugar de la búsqueda de texto simple. 
# 
# **Relaciones Clave:**
# 
# 1.  **Padre (Parent):** Un elemento que contiene a otros.
#     * *Ejemplo:* El `<article class="product_pod">` es el padre de toda la información de un libro.
# 2.  **Hijo (Child):** Un elemento contenido directamente dentro de su padre.
#     * *Ejemplo:* El `div.image_container` es un hijo directo del `<article>`.
# 3.  **Descendiente (Descendant):** Cualquier elemento contenido, sin importar el nivel de anidamiento.
#     * *Ejemplo:* La etiqueta `<a>` que tiene el título es descendiente del `<article>`, aunque no sea su hijo directo.
# 4.  **Hermano (Sibling):** Elementos que comparten el mismo padre.
#     * *Ejemplo:* En la estructura de un libro, el `div.image_container` y el `p.star-rating` son hermanos porque ambos están dentro del mismo `<article>`.
# 
# **Tu Trabajo como Scraper:**
# 
# El 90% del scraping es:
# 
# 1.  Encontrar un **contenedor maestro** que se repite (ej., `<article class="product_pod">`).
# 2.  Iterar sobre cada contenedor.
# 3.  Dentro de cada contenedor, usar **selectores CSS** (que usan las clases) para saltar directamente al dato de interés (ej., `p.price_color`).
# 
# ---

In [17]:
# Solo creamos la sopa si tenemos contenido para evitar errores
if html_content:
    soup = BeautifulSoup(html_content, "html.parser") # 'lxml' es un parser rápido y robusto
    print("¡Sopa creada exitosamente!")
else:
    print("No se puede crear la Sopa. El contenido HTML es nulo.")
    # Usamos una sopa de demostración para el resto de la conferencia si falla la conexión
    soup = BeautifulSoup('<html><body><div id="demo"><h1>Título Demo</h1><p class="data">Contenido</p></div></body></html>', 'lxml')

¡Sopa creada exitosamente!


# Parte 3: La Herramienta - `BeautifulSoup`
# 
# Analicemos cómo extraer datos de los libros:

#  A. Métodos de Búsqueda por Atributos y Etiquetas: `find` y `find_all`
# 
# Buscamos por **nombre de etiqueta** y **atributos exactos**.

In [18]:
# Ejemplo 1: Encontrar el primer <h1> y su texto
# El <h1> contiene el título de la página
primer_h1 = soup.find('h1')
print(f"find('h1'): {primer_h1.text.strip()}\n")

# Ejemplo 2: Encontrar la primera tarjeta de libro (etiqueta <article> con clase 'product_pod')
# Esto nos dará el contenedor del primer libro.
primer_libro_contenedor = soup.find('article', class_='product_pod')

# Ejemplo 3: Encontrar TODOS los contenedores de libros por CLASE
todos_los_libros_elementos = soup.find_all('article', class_='product_pod')
print(f"find_all('article', class_='product_pod'): Se encontraron {len(todos_los_libros_elementos)} libros.")

AttributeError: 'NoneType' object has no attribute 'text'

#  B. Accediendo a Atributos (La Extracción del Título)
# 
# El título del libro está en el atributo `title` de la etiqueta `<a>`.

In [ ]:
# Ejemplo 4: Obtener el título del primer libro
if primer_libro_contenedor:
    # 1. Buscar el link <a> dentro del contenedor del libro
    titulo_link_elem = primer_libro_contenedor.find('a', title=True) 
    
    # 2. Acceder al atributo 'title' como un diccionario
    if titulo_link_elem:
        titulo = titulo_link_elem['title'] 
        print(f"Título del primer libro (atributo 'title'): {titulo}")
    else:
        print("No se encontró el título.")
else:
    print("Asegúrate de que 'primer_libro_contenedor' fue definido exitosamente.")

# C. El Método Avanzado: Selectores CSS (`select` y `select_one`)
# 
# Usamos la sintaxis CSS para búsquedas complejas: **`etiqueta.clase`** o **`etiqueta > etiqueta`**.

In [ ]:
# Ejemplo 5: Encontrar el precio de TODOS los libros
# Selector: La etiqueta <p> con clase 'price_color'
precios_elementos = soup.select('p.price_color')
print(f"\nselect('p.price_color'): Se encontraron {len(precios_elementos)} precios.")
print(f"El precio del primer libro es: {precios_elementos[0].text.strip()}")

# Ejemplo 6: Búsqueda anidada y de Calificación
# Queremos la calificación de estrellas, que está en <p class="star-rating [Calificacion]">
# El selector es 'p.star-rating'
calificacion_elem = soup.select_one('p.star-rating')

if calificacion_elem:
    # La calificación (ej. 'Three') está en el segundo elemento de la lista de clases
    clases = calificacion_elem['class'] 
    calificacion = clases[1] # [1] porque [0] es 'star-rating' y [1] es 'Three', 'Two', etc.
    print(f"La calificación del primer libro (clase CSS) es: {calificacion}")
else:
    print("No se encontró la calificación.")


#  D. Navegando el Árbol (Tree Traversal)
# 
# Moverse desde un elemento conocido a su padre o hermano.

In [ ]:
# Ejemplo 7: Navegación del Árbol (Ir del precio al título)
# Partimos del primer precio
if precios_elementos:
    precio_elem = precios_elementos[0]
    
    # 1. ¿Quién es mi abuelo (el contenedor principal del libro)?
    # precio -> <div>.price_color -> <article>.product_pod
    contenedor_libro = precio_elem.parent.parent 
    
    # 2. Ahora, dentro de ese abuelo, buscamos el título
    titulo_elem = contenedor_libro.select_one('h3 a') 
    
    if titulo_elem:
        print(f"\nPartimos del precio, encontramos el abuelo, y extrajimos el título: {titulo_elem['title']}")
    else:
        print("Fallo en la navegación del árbol: no se encontró el título.")
else:
    print("Asegúrate de que la lista de precios no esté vacía.")

#  Parte 4: El Proyecto - Scraper de "Books to Scrape"
# 
# Aplicaremos la técnica `select` y la extracción de atributos para crear nuestro dataset.

In [ ]:
# 1. Creamos la lista vacía para guardar los datos
lista_de_libros_final = []

# 2. Encontramos todos los contenedores de libros
contenedores_libros = soup.select('article.product_pod')
print(f"\n--- Iniciando Proyecto: Extracción de {len(contenedores_libros)} libros ---")

# 3. Iteramos sobre cada contenedor y aplicamos selectores CSS anidados
for libro_contenedor in contenedores_libros:
    
    # a) Extraer Título (Atributo 'title' del link <a> dentro de <h3>)
    titulo_elem = libro_contenedor.select_one('h3 a')
    titulo = titulo_elem['title'].strip() if titulo_elem and 'title' in titulo_elem.attrs else 'N/A'
    
    # b) Extraer Precio (Texto del elemento <p.price_color>)
    precio_elem = libro_contenedor.select_one('p.price_color')
    precio = precio_elem.text.strip() if precio_elem else 'N/A'
    
    # c) Extraer Calificación (Segundo elemento de la lista de clases en <p.star-rating>)
    calificacion_elem = libro_contenedor.select_one('p.star-rating')
    clases = calificacion_elem['class'] if calificacion_elem and 'class' in calificacion_elem.attrs else []
    # El segundo elemento de la lista de clases es la calificación (ej. 'Five')
    calificacion = clases[1] if len(clases) > 1 else 'N/A'
    
    # d) Extraer URL de la imagen (Atributo 'src' de la etiqueta <img>)
    imagen_elem = libro_contenedor.select_one('div.image_container img')
    imagen_url = imagen_elem['src'].strip() if imagen_elem and 'src' in imagen_elem.attrs else 'N/A'
    
    # 4. Guardamos todo en un diccionario
    libro_dict = {
        'titulo': titulo,
        'precio': precio,
        'calificacion_estrellas': calificacion,
        'imagen_url': imagen_url
    }
    
    # 5. Añadimos el diccionario a nuestra lista maestra
    lista_de_libros_final.append(libro_dict)

# ¡Revisamos el resultado!
print("--- Ejemplo del primer libro extraído ---")
if lista_de_libros_final:
    print(json.dumps(lista_de_libros_final[0], indent=2, ensure_ascii=False))
else:
    print("Lista vacía. Hubo un error en la obtención del HTML o en el parseo.")


# Parte 5: La Persistencia (Conexión con Ciencia de Datos)
# 
# Convertimos nuestra lista de diccionarios en un `DataFrame` de `pandas` y lo guardamos.

In [ ]:
# Creamos el DataFrame
df = pd.DataFrame(lista_de_libros_final)

# ¡Vista Previa!
print("\n--- Vista Previa del DataFrame ---")
print(df.head())

#%% [code]
# El paso final: Guardar nuestro dataset
try:
    df.to_csv('libros_scrapeados.csv', index=False, encoding='utf-8')
    print("\n¡DataFrame guardado exitosamente en 'libros_scrapeados.csv'!")
except Exception as e:
    print(f"Error al guardar el CSV: {e}")

# Reto 
# 
# **¡Lo logramos!** Hemos reforzado la teoría del DOM, hecho que `requests` sea robusto con el **User-Agent**, dominado los métodos de `BeautifulSoup` y creado un dataset de libros.
# 
# **Reto (El Scraper Completo):**
# 
# Solo scrapeamos la primera página. Observa el botón "Next":
# 
# 1.  Inspecciona el botón "Next". Es un `<li class="next">` que contiene un link `<a>`.
# 2.  ¿Puedes obtener el `href` de ese link? (Ej: `catalogue/page-2.html`)
# 3.  ¿Puedes crear un bucle (`while`) que siga scrapeando mientras exista un botón "Next" y añadiendo los resultados a la lista?
#     * **Pista:** Necesitarás construir la URL completa, ya que el `href` es relativo (ej: `http://books.toscrape.com/` + `catalogue/page-2.html`).
# 
.